# BLS survey discovery

Enumerate BLS surveys and export series metadata catalogs to `data/`.

Source of truth is the BLS flat-file site (`download.bls.gov`), not the API: the
API cannot enumerate series and exposes neither coverage dates nor index bases.
Observations still come from the API via `fetch_series`.

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append('../')

from lib.utils import print_json_vertical
from utils import (
    CORE_SURVEYS,
    call_tool,
    list_mcp_tools,
    show_all_surveys,
    popular_series,
    fetch_bls_source_files,
    fetch_popular_ids,
    export_oe_national,
    write_survey_yaml,
    write_all_series_yaml,
    fetch_series,
)

## List every survey

In [2]:
surveys = await show_all_surveys()

AP: Consumer Price Index - Average Price Data
BD: Business Employment Dynamics
BG: Collective Bargaining Agreements-State and Local Government
BP: Collective Bargaining Agreements-Private Sector
CA: Biennial Nonfatal Case and Demographic numbers and rates: selected characteristics
CB: Biennial Nonfatal Case and Demographic numbers and rates: selected characteristics
CC: Employer Costs for Employee Compensation
CD: Nonfatal cases involving days away from work: selected characteristics
CE: Employment, Hours, and Earnings from the Current Employment Statistics survey (National)
CF: Census of Fatal Occupational Injuries
CH: Nonfatal cases involving days away from work: selected characteristics (2003 - 2010)
CI: Employment Cost Index
CM: Employer Costs for Employee Compensation
CS: Nonfatal cases involving days away from work: selected characteristics (2011 forward)
CU: Consumer Price Index - All Urban Consumers
CW: Consumer Price Index - Urban Wage Earners and Clerical Workers
CX: Consumer

In [3]:
survey_abbreviations = [survey['survey_abbreviation'] for survey in surveys]
survey_abbreviations

['AP',
 'BD',
 'BG',
 'BP',
 'CA',
 'CB',
 'CC',
 'CD',
 'CE',
 'CF',
 'CH',
 'CI',
 'CM',
 'CS',
 'CU',
 'CW',
 'CX',
 'EB',
 'EC',
 'EE',
 'EI',
 'EN',
 'EP',
 'EW',
 'FA',
 'FI',
 'FM',
 'FW',
 'GG',
 'GP',
 'HC',
 'HS',
 'II',
 'IN',
 'IP',
 'IS',
 'JL',
 'JT',
 'KV',
 'LA',
 'LE',
 'LF',
 'LI',
 'LN',
 'LU',
 'ML',
 'MP',
 'MU',
 'MW',
 'NB',
 'NC',
 'ND',
 'NW',
 'OE',
 'OR',
 'PC',
 'PD',
 'PF',
 'PI',
 'PR',
 'SA',
 'SH',
 'SI',
 'SM',
 'SU',
 'TU',
 'WD',
 'WM',
 'WP',
 'WS']

## Export popular-series catalogs for selected surveys

`LN` = Labor Force Statistics (CPS), `CU` = CPI-U, `CE` = Current Employment Statistics, `LA` = Local Area Unemployment.

In [ ]:
# Generate the metadata catalog. The fetch is ~75 min (gentle); to test
# quickly, swap CORE_SURVEYS for a single survey, e.g. ["AP"].
manifest = await fetch_bls_source_files(CORE_SURVEYS, delay=3.0)  # 22 core surveys
popular = await fetch_popular_ids(CORE_SURVEYS)                    # is_popular flags
await export_oe_national()                                        # OE national slice (streams ~1.26 GB)

write_survey_yaml(manifest=manifest)                              # -> data/survey.yaml (all 23)
write_all_series_yaml(CORE_SURVEYS, popular=popular)              # -> data/bls_series_<CODE>.yaml

## Fetch observations for a chosen set of series

In [6]:
await fetch_series(
    ["LNS14000000", "CUUR0000SA0", "CES0000000001"],
    "series_data/bls_headline.yaml",
    start_year=2015,
    end_year=2024,
    calculations=True,
)

Fetched 3 series (total 3)
Wrote 3 series to series_data/bls_headline.yaml
